# Phase 4C: independent paired-dev audit
Use the same Kaggle notebook as training/inference, but save a new version. Attach the exact successful paired-inference output from version 5, the original raw CSV and the phase-3 SRO dataset. Use CPU (accelerator None).


In [1]:
"""Independent CPU audit of the frozen paired-dev Kaggle inference output."""

import csv
import hashlib
import json
from pathlib import Path


BASE = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-gap-deberta-full-musique/fg_full_data")
SRO = Path("/kaggle/input/datasets/hungluongduc/fact-grounding-sro-phase-3-token-audit-20261121")
PRED_ROOT = Path("/kaggle/input/notebooks/hungluongduc/fact-grounding-sro-phase-4c-epoch-3-retry/phase4c_paired_dev")
OUT = Path("/kaggle/working/phase4c_paired_audit")
AMBIGUOUS_INDICES = [333, 335, 341, 343, 345, 350, 358, 375, 398, 426, 455, 459, 460, 461, 463, 543, 547, 551, 563, 650, 701, 703, 705, 707, 709, 711, 713, 715, 748, 854, 870, 874, 1035, 1037, 1066, 1078, 1136, 1138, 1140, 1188, 1254, 1300, 1304, 1310, 1333, 1337, 1339, 1343, 1378, 1382, 1399, 1400, 1439, 1447, 1449, 1453, 1455, 1459, 1475, 1477, 1479, 1481, 1483, 1484, 1487, 1488, 1531, 1534, 1537, 1539, 1543, 1545, 1547, 1549, 1574, 1580, 1600, 1616, 1656, 1673, 1675, 1677, 1679, 1681, 1683, 1687, 1723, 1729, 1743, 1747, 1749, 1757, 1838, 1840, 1848, 1857, 1861, 1869, 1873, 1875, 1877, 1879, 1883, 1975, 1976, 2000, 2002, 2016, 2018, 2025, 2033, 2035, 2037, 2039, 2043, 2045, 2047, 2056, 2058, 2062, 2066, 2121, 2125, 2163, 2165, 2171, 2173, 2175, 2179, 2183, 2185, 2197, 2199, 2213, 2215, 2219, 2412, 2418, 2433, 2531, 2534, 2537, 2540, 2543, 2546, 2549, 2552, 2555, 2600, 2603, 2606, 2609, 2612, 2615, 2618, 2621, 2624, 2627, 2630, 2633, 2636, 2639, 2642, 2645, 2648, 2651, 2654, 2657, 2660, 2663, 2730, 2736, 2819, 2822, 2828, 2840, 2843, 2862, 2902, 2903, 2907, 2909, 2917, 2918, 2923, 2929, 2930, 2935, 2936, 2941, 2944, 2945, 2949, 2951, 2953, 2958, 2960, 2962, 2964, 2966, 2975, 2980, 2981, 2989, 2990, 2995, 2996, 3008, 3013, 3014, 3018, 3020, 3024, 3026, 3028, 3029, 3035, 3036, 3038, 3040, 3041, 3044, 3045, 3047, 3049, 3050, 3052, 3053, 3055, 3056, 3057, 3059, 3062, 3065, 3071, 3074, 3096, 3126, 3135, 3202, 3211, 3214, 3221, 3230, 3239, 3242, 3245, 3269, 3275, 3278, 3305, 3314, 3329, 3410, 3413, 3419, 3422, 3425, 3428, 3431, 3434, 3437, 3443, 3446, 3449, 3452, 3455, 3458, 3461, 3464, 3470, 3472, 3473, 3477, 3479, 3500, 3617, 3618, 3627, 3633, 3642, 3645, 3651, 3663, 3675, 3681, 3690, 3696, 3698, 3705, 3711, 3723, 3729, 3733, 3747, 3753, 3762, 3771, 3774, 3789, 3890, 3946, 3950, 3959, 3962, 3965, 3968, 3971, 3974, 3983, 4038, 4059, 4224, 4311, 4315, 4320, 4324, 4328, 4380, 4384, 4403, 4447, 4460, 4461, 4464, 4468, 4470, 4472, 4474, 4476, 4478, 4480, 4484, 4486, 4488, 4492, 4496, 4500, 4504, 4506, 4508, 4512, 4516, 4518, 4520, 4522, 4524, 4528, 4530, 4532, 4534, 4536, 4538, 4540, 4544, 4548, 4564, 4568, 4612, 4616, 4620, 4624, 4628, 4636, 4640, 4644, 4705, 4730, 4734, 4746, 4750, 4754, 4762, 4800, 4804, 4808, 4812, 4816, 4820, 4825, 4828, 4832, 4836, 4840, 4844, 4848, 4851, 4853, 4856, 4860, 4864, 4868, 4871, 4872, 4876, 4882, 4883, 4885, 4889, 4895, 4899, 4903, 4907, 4911, 4915, 4919, 4923, 4927, 4930, 4931, 4935, 4939, 4943, 4947, 4951, 4955, 4959, 4963, 4967, 4971, 4975, 5079, 5083, 5088, 5095, 5096, 5103, 5115, 5119, 5139, 5151, 5155, 5156, 5160, 5167, 5171, 5254, 5258, 5262, 5266, 5270, 5274, 5278, 5282, 5286, 5290, 5294, 5298, 5302, 5306, 5310, 5314, 5318, 5322, 5326, 5330, 5334, 5338, 5342, 5486, 5490, 5494, 5498, 5502, 5506, 5510, 5514, 5518, 5522, 5526, 5557, 5593, 5596, 5602, 5606, 5610, 5618, 5620, 5626, 5628, 5638, 5687, 5777, 5785, 5788, 5801, 5821, 5825, 5829, 5837, 5841, 5849, 5878, 5885, 5893, 5897, 5901, 5912, 5930, 5936, 5943, 5955, 5963, 5973, 5975, 5999, 6000, 6016, 6025, 6026, 6041, 6049, 6058, 6077, 6092, 6116, 6118, 6143, 6151, 6166, 6173, 6185, 6188, 6193, 6215, 6217, 6256, 6283, 6299, 6335, 6338, 6346, 6347, 6352, 6362, 6373, 6380]
FIELDS = ["row_index", "label", "raw_text_sha256", "sro_text_sha256",
          "raw_prob", "sro_prob", "raw_pred", "sro_pred"]


def rows(path):
    with path.open(encoding="utf-8-sig", newline="") as file:
        reader = csv.DictReader(file)
        return reader.fieldnames, list(reader)


def digest(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest().upper()


def metric(y, p):
    tp = sum(a == 1 and b == 1 for a, b in zip(y, p))
    fp = sum(a == 0 and b == 1 for a, b in zip(y, p))
    fn = sum(a == 1 and b == 0 for a, b in zip(y, p))
    tn = len(y) - tp - fp - fn
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    return {"rows": len(y), "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "precision": precision, "recall": recall,
            "f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
            "accuracy": (tp + tn) / len(y),
            "flag_rate_prediction_0": (tn + fn) / len(y)}


def main():
    raw_fields, raw = rows(BASE / "deberta_fg_dev.csv")
    sro_fields, sro = rows(SRO / "dev.csv")
    pred_fields, pred = rows(PRED_ROOT / "paired_predictions.csv")
    original = json.loads((PRED_ROOT / "metrics.json").read_text(encoding="utf-8"))
    if raw_fields != ["text", "label"] or sro_fields != ["text", "label"] or pred_fields != FIELDS:
        raise ValueError("CSV schema mismatch")
    if (len(raw), len(sro), len(pred)) != (6404, 6404, 6404):
        raise ValueError("paired row count mismatch")
    if AMBIGUOUS_INDICES is None:
        raise ValueError("frozen phase-3 status indices missing")
    ambiguous = set(AMBIGUOUS_INDICES)
    if len(ambiguous) != len(AMBIGUOUS_INDICES) or any(i < 0 or i >= 6404 for i in ambiguous):
        raise ValueError("invalid status indices")
    labels, raw_preds, sro_preds = [], [], []
    for i, (a, b, p) in enumerate(zip(raw, sro, pred)):
        if int(p["row_index"]) != i or a["label"] != b["label"] or p["label"] != a["label"]:
            raise ValueError(f"index or label mismatch at row {i}")
        if digest(a["text"]) != p["raw_text_sha256"] or digest(b["text"]) != p["sro_text_sha256"]:
            raise ValueError(f"text hash mismatch at row {i}")
        for branch in ("raw", "sro"):
            probability = float(p[f"{branch}_prob"])
            if not 0 <= probability <= 1 or int(p[f"{branch}_pred"]) != int(probability >= 0.5):
                raise ValueError(f"invalid probability or threshold at row {i}, {branch}")
        labels.append(int(a["label"]))
        raw_preds.append(int(p["raw_pred"]))
        sro_preds.append(int(p["sro_pred"]))
    recalculated = {"raw": metric(labels, raw_preds), "sro": metric(labels, sro_preds)}
    if recalculated != {key: original[key] for key in ("raw", "sro")}:
        raise ValueError("independent metrics disagree with inference report")
    disagreements = [i for i in range(6404) if raw_preds[i] != sro_preds[i]]
    if len(disagreements) != original["paired_disagreement_rows"]:
        raise ValueError("disagreement count mismatch")
    if original["coverage"] != {"ok": 6404 - len(ambiguous), "ambiguous_fallback": len(ambiguous)}:
        raise ValueError("SRO coverage mismatch")
    prediction_path = PRED_ROOT / "paired_predictions.csv"
    prediction_sha256 = hashlib.sha256(prediction_path.read_bytes()).hexdigest().upper()
    if prediction_sha256 != original["predictions_sha256"]:
        raise ValueError("prediction file hash mismatch")
    report = {"status": "passed", "dev_rows": 6404, "predictions_sha256": prediction_sha256,
              "raw": recalculated["raw"], "sro": recalculated["sro"],
              "sro_minus_raw_f1": recalculated["sro"]["f1"] - recalculated["raw"]["f1"],
              "paired_disagreement_rows": len(disagreements),
              "coverage": original["coverage"],
              "caveat": "Validation grouped by normalized sub-question proxy; parent qid unverified."}
    if OUT.exists():
        raise FileExistsError(OUT)
    OUT.mkdir(parents=True)
    (OUT / "audit.json").write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)


if __name__ == "__main__":
    main()


{
  "status": "passed",
  "dev_rows": 6404,
  "predictions_sha256": "2F794986461445DA6AEAAFE632DA378205CFD663A87CE1FC1815A08F8D458AB8",
  "raw": {
    "rows": 6404,
    "tp": 2299,
    "fp": 387,
    "fn": 846,
    "tn": 2872,
    "precision": 0.8559195830230827,
    "recall": 0.7310015898251192,
    "f1": 0.7885439890241812,
    "accuracy": 0.8074640849469081,
    "flag_rate_prediction_0": 0.5805746408494691
  },
  "sro": {
    "rows": 6404,
    "tp": 2237,
    "fp": 507,
    "fn": 908,
    "tn": 2752,
    "precision": 0.8152332361516035,
    "recall": 0.7112877583465819,
    "f1": 0.7597215146884021,
    "accuracy": 0.7790443472829481,
    "flag_rate_prediction_0": 0.5715178013741412
  },
  "sro_minus_raw_f1": -0.028822474335779114,
  "paired_disagreement_rows": 726,
  "coverage": {
    "ok": 5867,
    "ambiguous_fallback": 537
  },
  "caveat": "Validation grouped by normalized sub-question proxy; parent qid unverified."
}
